# Notebook 03 — Model A: TF-IDF + Linear Baseline
**Week 2**: One-vs-rest Logistic Regression / LinearSVC trained on TF-IDF features with per-label threshold tuning.

In [ ]:
# pip install -q scikit-learn pandas pyarrow numpy

In [ ]:
# Running locally — no Drive mount needed

## 1. Config

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import pickle, os, json
from pathlib import Path

DATA_DIR  = '../data/datasets'
MODEL_DIR = '../data/models/model_a'
os.makedirs(MODEL_DIR, exist_ok=True)

CLASSIFIER = 'lr'
C_REG      = 1.0
MAX_ITER   = 1000
N_JOBS     = -1

## 2. Load features & labels

In [ ]:
X_train = sp.load_npz(f'{DATA_DIR}/X_train_tfidf.npz')
X_val   = sp.load_npz(f'{DATA_DIR}/X_val_tfidf.npz')
X_test  = sp.load_npz(f'{DATA_DIR}/X_test_tfidf.npz')
Y_train = np.load(f'{DATA_DIR}/Y_train.npy')
Y_val   = np.load(f'{DATA_DIR}/Y_val.npy')
Y_test  = np.load(f'{DATA_DIR}/Y_test.npy')

with open(f'{DATA_DIR}/mlb.pkl', 'rb') as f:
    mlb = pickle.load(f)
vocab = list(mlb.classes_)

print(f'X_train: {X_train.shape}  Y_train: {Y_train.shape}')
print(f'Labels: {len(vocab)}')

## 3. Train One-vs-Rest classifier

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

if CLASSIFIER == 'lr':
    base = LogisticRegression(C=C_REG, max_iter=MAX_ITER, solver='saga',
                              class_weight='balanced', n_jobs=N_JOBS)
    clf = OneVsRestClassifier(base, n_jobs=N_JOBS)
else:  # LinearSVC — needs CalibratedClassifierCV for probability estimates
    base = CalibratedClassifierCV(
        LinearSVC(C=C_REG, max_iter=MAX_ITER, class_weight='balanced'),
        cv=3, method='sigmoid'
    )
    clf = OneVsRestClassifier(base, n_jobs=N_JOBS)

print(f'Training {CLASSIFIER} OvR on {X_train.shape[0]} samples × {len(vocab)} labels...')
clf.fit(X_train, Y_train)
print('Done.')

with open(f'{MODEL_DIR}/clf_{CLASSIFIER}.pkl', 'wb') as f:
    pickle.dump(clf, f)
print('Model saved.')

## 4. Probability estimates

In [ ]:
print('Computing probabilities on val set...')
P_val  = clf.predict_proba(X_val)   # shape (n_val, n_labels)
P_test = clf.predict_proba(X_test)
print(f'P_val shape: {P_val.shape}')

## 5. Threshold tuning on validation set

In [ ]:
from sklearn.metrics import f1_score

def tune_global_threshold(P, Y, thresholds=np.arange(0.05, 0.55, 0.025)):
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        preds = (P >= t).astype(int)
        f1 = f1_score(Y, preds, average='micro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

best_t, best_f1_val = tune_global_threshold(P_val, Y_val)
print(f'Best global threshold: {best_t:.3f}  →  micro-F1 on val: {best_f1_val:.4f}')

## 6. Evaluation

In [ ]:
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score, roc_auc_score
)

def evaluate(P, Y, threshold, split_name='test'):
    preds = (P >= threshold).astype(int)
    results = {
        'split'       : split_name,
        'threshold'   : threshold,
        'micro_f1'    : f1_score(Y, preds, average='micro',   zero_division=0),
        'macro_f1'    : f1_score(Y, preds, average='macro',   zero_division=0),
        'micro_prec'  : precision_score(Y, preds, average='micro', zero_division=0),
        'micro_rec'   : recall_score(Y, preds, average='micro',    zero_division=0),
    }
    # AUPRC / AUROC — only for labels with positives in this split
    mask = Y.sum(0) > 0
    results['macro_auprc'] = average_precision_score(Y[:, mask], P[:, mask], average='macro')
    results['micro_auroc'] = roc_auc_score(Y[:, mask], P[:, mask], average='micro')
    for k, v in results.items():
        if isinstance(v, float):
            print(f'  {k:20s}: {v:.4f}')
        else:
            print(f'  {k:20s}: {v}')
    return results

print('=== Validation ===')
val_results  = evaluate(P_val,  Y_val,  best_t, 'val')
print('\n=== Test ===')
test_results = evaluate(P_test, Y_test, best_t, 'test')

with open(f'{MODEL_DIR}/results.json', 'w') as f:
    json.dump({'val': val_results, 'test': test_results}, f, indent=2)
print('\nResults saved.')

## 7. Head vs. tail label analysis

In [ ]:
import matplotlib.pyplot as plt

# Per-label F1 on test set
preds_test = (P_test >= best_t).astype(int)
per_label_f1   = f1_score(Y_test, preds_test, average=None, zero_division=0)
per_label_freq = Y_train.sum(0)   # training frequency

label_df = pd.DataFrame({
    'icd_code'   : vocab,
    'train_freq' : per_label_freq,
    'test_f1'    : per_label_f1
}).sort_values('train_freq', ascending=False)

# Bucket labels into head / torso / tail
thresholds_freq = [500, 100, 0]
labels_bucket   = ['head (≥500)', 'torso (100-499)', 'tail (<100)']

for bucket, (lo, hi) in zip(labels_bucket, [(500, 1e9), (100, 499), (0, 99)]):
    subset = label_df[(label_df['train_freq'] >= lo) & (label_df['train_freq'] <= hi)]
    print(f'{bucket:25s}  n_codes={len(subset):5d}  avg_F1={subset["test_f1"].mean():.4f}')

# Scatter: frequency vs F1
plt.figure(figsize=(7, 4))
plt.scatter(label_df['train_freq'].clip(upper=2000), label_df['test_f1'], alpha=0.3, s=8)
plt.xscale('log'); plt.xlabel('Train freq (log)'); plt.ylabel('Test F1')
plt.title('Per-label F1 vs training frequency (Model A)')
plt.tight_layout(); plt.savefig(f'{MODEL_DIR}/head_tail_f1.png', dpi=120); plt.show()

## 8. Top-K evaluation (Top-50, Top-100, Top-500)

In [ ]:
for K in [50, 100, 500]:
    top_k_idx = label_df.nlargest(K, 'train_freq').index
    Y_k = Y_test[:, top_k_idx]
    P_k = P_test[:, top_k_idx]
    preds_k = (P_k >= best_t).astype(int)
    mf1 = f1_score(Y_k, preds_k, average='micro', zero_division=0)
    print(f'Top-{K:4d}  micro-F1 = {mf1:.4f}')